In [ ]:
# ==========================================
# 1. INSTALL NECESSARY TOOLS
# ==========================================
!pip install gdown flask-ngrok pyngrok transformers torch tensorflow joblib Pillow

import os
import gdown
import torch
import numpy as np
import tensorflow as tf
import joblib
import base64
import io
from PIL import Image
from flask import Flask, request, jsonify
from pyngrok import ngrok
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# ==========================================
# 2. CONFIGURATION & AUTH
# ==========================================
NGROK_TOKEN = "Add Token"
ngrok.set_auth_token(NGROK_TOKEN)

app = Flask(__name__)

# ==========================================
# 3. DOWNLOAD MODELS FROM DRIVE
# ==========================================
print("📥 Downloading Models from Google Drive...")

# Download Folders (These IDs match your public links)
!gdown --folder 1GnWWFhlsOkCirkmu9O_ElCKL_gSeysIo -O image_models/
!gdown --folder 19Yuhj6IqVrqoQ28TGo-q7CSF-bsb3Vam -O text_models/

print("✅ Download Complete!")

# ==========================================
# 4. LOAD MODELS INTO GPU
# ==========================================
print("🧠 Loading models into T4 GPU...")

# Path to Image Model
IMAGE_MODEL_PATH = 'image_models/Image_Models/symptom_image_model_v2.h5'
image_model = tf.keras.models.load_model(IMAGE_MODEL_PATH)

# Path to Heavy Text Model (DistilBERT v5)
TEXT_MODEL_PATH = 'text_models/heavy_distilbert_ensemble/v5'
text_model = DistilBertForSequenceClassification.from_pretrained(TEXT_MODEL_PATH).to("cuda")
tokenizer = DistilBertTokenizer.from_pretrained(TEXT_MODEL_PATH)

# Path to Label Encoder
LABEL_ENCODER_PATH = 'text_models/heavy_distilbert_ensemble/label_encoder.pkl'
le_text = joblib.load(LABEL_ENCODER_PATH)

print("🚀 All Models Live!")

# ==========================================
# 5. PRE-FLIGHT TEST (PROVE IT WORKS)
# ==========================================
print("\n🔍 --- RUNNING TESTS ---")

# --- Test 1: Text Prediction ---
test_text = "I have a severe headache, high fever, and a stiff neck."
inputs = tokenizer(test_text, return_tensors="pt", truncation=True, padding=True, max_length=128).to("cuda")
with torch.no_grad():
    outputs = text_model(**inputs)
    idx = torch.argmax(outputs.logits, dim=1).item()
text_result = le_text.inverse_transform([idx])[0]
print(f"✅ Text Model Test: '{test_text}' -> PREDICTION: {text_result}")

# --- Test 2: Image Prediction (Using a dummy image) ---
dummy_img = np.random.rand(1, 224, 224, 3).astype(np.float32)
img_preds = image_model.predict(dummy_img, verbose=0)
classes = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
img_result = classes[img_preds.argmax()]
print(f"✅ Image Model Test: [Random Pixels] -> PREDICTION: {img_result}")
print("------------------------\n")

# ==========================================
# 6. API ENDPOINTS (v2: confidence, top-k, /health, API key, threaded)
# ==========================================
import time
import torch.nn.functional as F

API_KEY = ""   # optional shared secret. If set, put the SAME value in webapp .env.local as KAGGLE_API_KEY
IMG_CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

@app.before_request
def check_key():
    if request.path == '/health':
        return None
    if API_KEY and request.headers.get('x-api-key') != API_KEY:
        return jsonify({"error": "unauthorized", "status": "error"}), 401

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "success", "text_model": True, "image_model": True, "time": time.time()})

@app.route('/predict-text', methods=['POST'])
def predict_text():
    try:
        data = request.get_json(force=True, silent=True) or {}
        text = (data.get('symptoms') or '').strip()
        if not text:
            return jsonify({"error": "symptoms is required", "status": "error"}), 400
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to("cuda")
        with torch.no_grad():
            logits = text_model(**inputs).logits
        probs = F.softmax(logits, dim=1)[0].cpu().numpy()
        order = probs.argsort()[::-1][:3]
        top_k = [{"label": le_text.inverse_transform([int(i)])[0], "confidence": float(probs[i])} for i in order]
        return jsonify({"illness": top_k[0]["label"], "confidence": top_k[0]["confidence"], "top_k": top_k, "status": "success"})
    except Exception as e:
        return jsonify({"error": str(e), "status": "error"}), 500

@app.route('/predict-image', methods=['POST'])
def predict_image():
    try:
        data = request.get_json(force=True, silent=True) or {}
        img_b64 = data.get('image', '')
        if not img_b64:
            return jsonify({"error": "image is required", "status": "error"}), 400
        img = Image.open(io.BytesIO(base64.b64decode(img_b64))).convert('RGB').resize((224, 224))
        arr = tf.keras.preprocessing.image.img_to_array(img) / 255.0
        arr = tf.expand_dims(arr, 0)
        preds = image_model.predict(arr, verbose=0)[0]
        order = preds.argsort()[::-1][:3]
        top_k = [{"label": IMG_CLASSES[int(i)], "confidence": float(preds[i])} for i in order]
        return jsonify({"condition": top_k[0]["label"], "confidence": top_k[0]["confidence"], "top_k": top_k, "status": "success"})
    except Exception as e:
        return jsonify({"error": str(e), "status": "error"}), 500

# ==========================================
# 7. START THE TUNNEL
# ==========================================
public_url = ngrok.connect(5000)
print(f"🌍 YOUR PUBLIC API URL IS: {public_url}")
print("\n--- READY TO RECEIVE REQUESTS ---")

app.run(port=5000, threaded=True)
